In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/kazakh-missing-words-challenge-for-wish/sample_submission.csv
/kaggle/input/competitions/kazakh-missing-words-challenge-for-wish/test_inputs.csv
/kaggle/input/competitions/kazakh-missing-words-challenge-for-wish/train.csv
/kaggle/input/datasets/mereybolat/datasets-acm/train_generated.csv
/kaggle/input/datasets/mereybolat/datasets-acm/sample_submission.csv
/kaggle/input/datasets/mereybolat/datasets-acm/test_inputs.csv


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import re
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla T4


In [3]:
TRAIN_PATH  = '/kaggle/input/datasets/mereybolat/datasets-acm/train_generated.csv'
TEST_PATH   = '/kaggle/input/datasets/mereybolat/datasets-acm/test_inputs.csv'
OUTPUT_PATH = '/kaggle/working/submission.csv'

MODEL_NAME  = 'FacebookAI/xlm-roberta-base'

MAX_LEN     = 128
BATCH_SIZE  = 32
EPOCHS      = 3
LR          = 2e-5
MAX_LABEL   = 64
N_SAMPLES = 250000

In [4]:
import os

# Смотрим что вообще есть в /kaggle/input/
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/kazakh-missing-words-challenge-for-wish/sample_submission.csv
/kaggle/input/competitions/kazakh-missing-words-challenge-for-wish/test_inputs.csv
/kaggle/input/competitions/kazakh-missing-words-challenge-for-wish/train.csv
/kaggle/input/datasets/mereybolat/datasets-acm/train_generated.csv
/kaggle/input/datasets/mereybolat/datasets-acm/sample_submission.csv
/kaggle/input/datasets/mereybolat/datasets-acm/test_inputs.csv


In [5]:
DATASET_NAME = 'datasets-acm'

TRAIN_PATH  = '/kaggle/input/datasets/mereybolat/datasets-acm/train_generated.csv'
TEST_PATH   = '/kaggle/input/datasets/mereybolat/datasets-acm/test_inputs.csv'
OUTPUT_PATH = '/kaggle/working/submission.csv'

In [6]:
df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)


print(f'Train: {len(df_train)} строк')


Train: 554353 строк


Cleaning dataset

In [7]:
def clean_text(text):
    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

In [8]:
df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

# Очищаем тексты
df_train['text'] = df_train['text'].apply(clean_text)
df_test['text']  = df_test['text'].apply(clean_text)

# Убираем дубликаты
df_train = df_train.drop_duplicates(subset='text').reset_index(drop=True)

# Обрезаем редкие большие позиции
df_train['label'] = df_train['label'].clip(upper=MAX_LABEL - 1)

# Берём 250k (или все если меньше)
if len(df_train) > 250000:
    df_train = df_train.sample(250000, random_state=42).reset_index(drop=True)

print(f'Train: {len(df_train)} строк')
print(f'Test:  {len(df_test)} строк')
print(f'Диапазон меток: {df_train["label"].min()} — {df_train["label"].max()}')

Train: 250000 строк
Test:  15000 строк
Диапазон меток: 0 — 31


In [9]:
print(f'Train: {len(df_train)} строк')

Train: 250000 строк


In [10]:
df_train.tail()

,text,label
249995,бірақ қатты жығылған бетінен қарай алмады,4
249996,осыған байланысты барлық алдын алу сақтау қажет,5
249997,мәскеудегі меншікті тілшіміз индира бегайдар қ...,5
249998,рымбаева роза қазақконцерт мемлекеттік концерт...,2
249999,осы мәселені тексеру керек аныққанығын,2


In [11]:
print(f'Размер train: {len(df_train)}')


Размер train: 250000


In [12]:
!pip install transformers -q

In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class KazakhDataset(Dataset):
    """
    Token Alignment Dataset.
    
    Для каждого предложения находим позиции первых токенов каждого слова.
    Это помогает модели понимать границы слов.
    
    Пример:
    Слова:   [бүгін, депутаттары, мақұлдады]
    Токены:  [бүг, ін, депу, тат, тары, мақұл, дады]
    Позиции первых токенов: [0, 2, 5]  (0-indexed после [CLS])
    """
    def __init__(self, texts, labels=None):
        self.texts  = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        words = text.split()

        # Токенизируем с word_ids для token alignment
        encoding = tokenizer(
            text,
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
            return_offsets_mapping=False
        )

        # Находим первый токен каждого слова через word_ids
        encoding_plain = tokenizer(
            text,
            max_length=MAX_LEN,
            truncation=True
        )
        word_ids = encoding_plain.word_ids()  # [None, 0, 0, 1, 1, 1, 2, ...]

        # word_token_starts[i] = позиция первого токена i-го слова
        word_token_starts = [-1] * MAX_LABEL
        seen = set()
        for token_idx, word_idx in enumerate(word_ids):
            if word_idx is not None and word_idx not in seen:
                if word_idx < MAX_LABEL:
                    word_token_starts[word_idx] = token_idx
                seen.add(word_idx)

        item = {
            'input_ids':         encoding['input_ids'].squeeze(),
            'attention_mask':    encoding['attention_mask'].squeeze(),
            'word_token_starts': torch.tensor(word_token_starts, dtype=torch.long),
        }
        if self.labels is not None:
            item['label'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [14]:
class MissingWordModel(nn.Module):
    """
    kaz-roberta + Token Alignment + улучшенный Classification Head

    Архитектура:
    Текст → kaz-roberta → [CLS] + mean pooling
                        + эмбеддинги первых токенов каждого слова
    → concat → Linear → GELU → Linear → позиция
    """
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.roberta   = AutoModel.from_pretrained(model_name)
        hidden_size    = self.roberta.config.hidden_size  # 768
        self.num_classes = num_classes

        # CLS (768) + mean (768) = 1536
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask, word_token_starts):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        hidden  = outputs.last_hidden_state  # (batch, seq_len, 768)

        # [CLS] токен
        cls_out  = hidden[:, 0, :]  # (batch, 768)

        # Mean pooling
        mask     = attention_mask.unsqueeze(-1).float()
        mean_out = (hidden * mask).sum(1) / mask.sum(1)  # (batch, 768)

        # Concat CLS + mean
        pooled = torch.cat([cls_out, mean_out], dim=-1)  # (batch, 1536)

        return self.classifier(pooled)  # (batch, num_classes)

подготовка даных

In [15]:
X = df_train['text'].values
y = df_train['label'].values

# 90/10 split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=42
)

train_dataset = KazakhDataset(X_train, y_train)
val_dataset   = KazakhDataset(X_val,   y_val)
test_dataset  = KazakhDataset(df_test['text'].values)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

Train: 225000 | Val: 25000 | Test: 15000


обучение

In [16]:
model = MissingWordModel(MODEL_NAME, num_classes=MAX_LABEL).to(DEVICE)
print(f'Параметров модели: {sum(p.numel() for p in model.parameters()):,}')

optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

# Label smoothing (улучшение 5)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

best_val_acc = 0

for epoch in range(EPOCHS):
    # ── TRAIN ──────────────────────────────────────────────────────
    model.train()
    total_loss, correct, total = 0, 0, 0

    for i, batch in enumerate(train_loader):
        input_ids         = batch['input_ids'].to(DEVICE)
        attention_mask    = batch['attention_mask'].to(DEVICE)
        word_token_starts = batch['word_token_starts'].to(DEVICE)
        labels            = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask, word_token_starts)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

        if i % 500 == 0:
            print(f'  Epoch {epoch+1} | Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f}')

    train_acc = correct / total

    # ── VALIDATION ─────────────────────────────────────────────────
    model.eval()
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids         = batch['input_ids'].to(DEVICE)
            attention_mask    = batch['attention_mask'].to(DEVICE)
            word_token_starts = batch['word_token_starts'].to(DEVICE)
            labels            = batch['label'].to(DEVICE)
            logits = model(input_ids, attention_mask, word_token_starts)
            preds  = logits.argmax(dim=-1)
            val_correct += (preds == labels).sum().item()
            val_total   += labels.size(0)

    val_acc  = val_correct / val_total
    avg_loss = total_loss / len(train_loader)

    print(f'\nEpoch {epoch+1}/{EPOCHS} | loss: {avg_loss:.4f} | train_acc: {train_acc:.4f} | val_acc: {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/kaggle/working/best_model.pt')
        print(f'  ✓ Лучшая модель сохранена (val_acc={val_acc:.4f})\n')

print(f'\nЛучший val_acc: {best_val_acc:.4f}')

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Параметров модели: 278,863,424
  Epoch 1 | Batch 0/7032 | Loss: 4.1687
  Epoch 1 | Batch 500/7032 | Loss: 3.0650
  Epoch 1 | Batch 1000/7032 | Loss: 2.7468
  Epoch 1 | Batch 1500/7032 | Loss: 2.8007
  Epoch 1 | Batch 2000/7032 | Loss: 2.8289
  Epoch 1 | Batch 2500/7032 | Loss: 2.8662
  Epoch 1 | Batch 3000/7032 | Loss: 2.4473
  Epoch 1 | Batch 3500/7032 | Loss: 2.6238
  Epoch 1 | Batch 4000/7032 | Loss: 2.5431
  Epoch 1 | Batch 4500/7032 | Loss: 2.2620
  Epoch 1 | Batch 5000/7032 | Loss: 2.2803
  Epoch 1 | Batch 5500/7032 | Loss: 2.8179
  Epoch 1 | Batch 6000/7032 | Loss: 2.5334
  Epoch 1 | Batch 6500/7032 | Loss: 2.6027
  Epoch 1 | Batch 7000/7032 | Loss: 2.2986

Epoch 1/3 | loss: 2.6703 | train_acc: 0.2294 | val_acc: 0.4236
  ✓ Лучшая модель сохранена (val_acc=0.4236)

  Epoch 2 | Batch 0/7032 | Loss: 2.3015
  Epoch 2 | Batch 500/7032 | Loss: 2.2611
  Epoch 2 | Batch 1000/7032 | Loss: 2.2433
  Epoch 2 | Batch 1500/7032 | Loss: 2.0674
  Epoch 2 | Batch 2000/7032 | Loss: 2.1268
  Epoch

In [17]:
model.load_state_dict(torch.load('/kaggle/working/best_model.pt'))
model.eval()

all_preds = []
with torch.no_grad():
    for batch in test_loader:
        input_ids         = batch['input_ids'].to(DEVICE)
        attention_mask    = batch['attention_mask'].to(DEVICE)
        word_token_starts = batch['word_token_starts'].to(DEVICE)
        logits = model(input_ids, attention_mask, word_token_starts)
        preds  = logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)

print(f'Предсказаний: {len(all_preds)}')
print(f'Пример: {all_preds[:10]}')

Предсказаний: 15000
Пример: [np.int64(2), np.int64(3), np.int64(7), np.int64(0), np.int64(2), np.int64(0), np.int64(2), np.int64(6), np.int64(3), np.int64(3)]


In [18]:
submission = pd.DataFrame({
    'ID': df_test['ID'],
    'missing_word_position': all_preds
})

submission = submission.sort_values('ID').reset_index(drop=True)
submission.to_csv(OUTPUT_PATH, index=False)

print(f'Submission сохранён: {OUTPUT_PATH}')
print(f'Строк: {len(submission)}')
submission.head(10)

Submission сохранён: /kaggle/working/submission.csv
Строк: 15000


,ID,missing_word_position
0,0,2
1,1,3
2,2,7
3,3,0
4,4,2
5,5,0
6,6,2
7,7,6
8,8,3
9,9,3


In [19]:
# Проверка формата
sample = pd.read_csv('/kaggle/input/competitions/kazakh-missing-words-challenge-for-wish/sample_submission.csv')
print(f'Форматы совпадают: {list(submission.columns) == list(sample.columns)}')
print(f'Строк совпадает: {len(submission) == len(sample)}')

Форматы совпадают: True
Строк совпадает: True


In [20]:
from IPython.display import FileLink
FileLink('submission.csv')

/kaggle/working/submission.csv